# 10 — OCR multi-mesin (Tahap 10, Fase 2)

**Belum pernah dijalankan.** Buka, sesuaikan sel pengaturan bila perlu, lalu Run All.

## Apa yang dikerjakan

Membaca ulang **3.482 citra JPG yang sama** dengan tiga mesin OCR, lalu menyimpan teks
dan **kotak posisi teks** tiap halaman:

| Mesin | Jalan di | Perkiraan |
|---|---|---|
| Tesseract 5 (`--oem 1 --psm 3`, konfigurasi QS-OCR) | CPU | 0,2–0,4 |
| EasyOCR | GPU (torch yang sudah ada) | 5,7 (CPU) |
| PaddleOCR, model PP-OCRv5 mobile | CPU, **venv terpisah** | 2,0 |

Kolom terakhir adalah detik per halaman pada **satu halaman sintetis di Mac**, saat
adapter diuji — hanya untuk orientasi, bukan perkiraan untuk laptop ini. Kecepatan sungguhannya diukur di bagian 3 pada `UJI_COBA` halaman pertama, dan perkiraan
total dicetak **sebelum** proses penuh dimulai.

Keras-OCR **tidak** dijalankan. Alasannya ada di catatan `13-ocr-multi-mesin`: pustaka itu
butuh TensorFlow dengan API Keras 2, yang tidak tersedia untuk Python 3.12.

## Kenapa PaddleOCR di venv terpisah

PaddleOCR membawa `opencv-contrib-python` dan puluhan dependensi lain. Kalau dipasang di
lingkungan utama, ia **bentrok dengan opencv milik EasyOCR** (keduanya menyediakan modul
`cv2`) dan bisa menggeser numpy dari 1.26.4, yang hasil Fase 1 bergantung padanya.
Notebook ini membuat `.venv-paddle/` sendiri (± 1 GB di disk) dan memanggil PaddleOCR
lewat subprocess. Lingkungan utama tidak tersentuh.

## Yang ditulis, dan yang tidak

- Ditulis: `data/interim/ocr/<mesin>/<id>.json`, `reports/ocr_*.csv`, `reports/figures/ocr_*.png`
- **Tidak ada file lama yang ditimpa.** Hasil OCR bisa dilanjutkan: kalau terputus, Run
  All lagi dan halaman yang sudah selesai dilewati.

## Pengaturan

In [ ]:
MESIN = ["tesseract", "easyocr", "paddleocr"]   # hapus yang tidak ingin dijalankan
PADDLE_MODEL = "mobile"    # "mobile" (cepat) | "default" (PP-OCRv6_medium, ± 3x lebih lambat)
UJI_COBA = 20              # halaman per mesin untuk mengukur kecepatan lebih dulu
JALANKAN_PENUH = True      # False = berhenti setelah uji coba

import json, os, shutil, subprocess, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
MANIFEST = ROOT / "data" / "processed" / "manifest.csv"
OUT = ROOT / "data" / "interim" / "ocr"
assert MANIFEST.exists(), "manifest.csv belum ada — jalankan notebook 01 dulu."
print("root:", ROOT)

## 1. Dependency

Lingkungan utama mendapat `pytesseract`, `easyocr`, dan `wordfreq` (daftar kata untuk
mengukur OOV). Semuanya dipasang dengan **constraint** yang mengunci numpy dan torch pada
versi yang sudah terpasang. Tanpa constraint, `easyocr` menarik opencv terbaru yang
menuntut numpy 2 — dan pip akan diam-diam meng-upgrade numpy.

In [ ]:
from importlib import metadata

kunci = ROOT / "data" / "interim" / "constraints_ocr.txt"
kunci.parent.mkdir(parents=True, exist_ok=True)
kunci.write_text("\n".join(f"{p}=={metadata.version(p)}"
                           for p in ["numpy", "torch", "torchvision"]) + "\n")
print(kunci.read_text())
%pip install -q pytesseract easyocr wordfreq -c "{kunci}"

r = subprocess.run([sys.executable, "-c", "import numpy, cv2, easyocr;"
                    "print(numpy.__version__, cv2.__version__, easyocr.__version__)"],
                   capture_output=True, text=True)
print("numpy / opencv / easyocr:", r.stdout.strip() or r.stderr.strip()[-300:])
assert r.stdout.startswith(metadata.version("numpy")), "numpy berubah! Hentikan dan laporkan."

### Tesseract

Tesseract bukan paket pip, melainkan program terpisah. Di Windows sel ini memasangnya
lewat **winget** (build UB Mannheim), yang akan memunculkan jendela izin administrator —
klik *Yes*. Perintah winget di bawah menyetujui perjanjian lisensi paket atas nama Anda
(`--accept-package-agreements`); Tesseract berlisensi Apache-2.0.

Versinya Tesseract 5, bukan 4.0-beta seperti QS-OCR. Selisih Tesseract-JPG vs QS-OCR
karena itu mengukur **efek gabungan** format citra (JPG vs TIF) dan versi Tesseract.

In [ ]:
TESSERACT_CMD = shutil.which("tesseract")
WIN_PATH = Path(r"C:\Program Files\Tesseract-OCR\tesseract.exe")

if "tesseract" in MESIN and not TESSERACT_CMD:
    if WIN_PATH.exists():
        TESSERACT_CMD = str(WIN_PATH)
    elif sys.platform == "win32":
        print("Memasang Tesseract lewat winget...")
        subprocess.run(["winget", "install", "--id", "UB-Mannheim.TesseractOCR", "-e",
                        "--accept-source-agreements", "--accept-package-agreements"])
        TESSERACT_CMD = str(WIN_PATH) if WIN_PATH.exists() else None
    if not TESSERACT_CMD:
        raise RuntimeError(
            "Tesseract tidak ditemukan. Windows: pasang dari "
            "https://github.com/UB-Mannheim/tesseract/wiki lalu jalankan sel ini lagi. "
            "macOS: brew install tesseract. Linux: apt install tesseract-ocr. "
            "Atau hapus 'tesseract' dari MESIN.")

if "tesseract" in MESIN:
    print("tesseract:", TESSERACT_CMD)
    print(subprocess.run([TESSERACT_CMD, "--version"], capture_output=True,
                         text=True).stdout.splitlines()[0])

### PaddleOCR (venv terpisah)

Versi dikunci pada kombinasi yang sudah diuji: `paddlepaddle==3.3.1` (CPU) dan
`paddleocr==3.7.0`. Membuat venv dan memasangnya memakan beberapa menit; sekali saja.

In [ ]:
VENV_PADDLE = ROOT / ".venv-paddle"
PY_PADDLE = VENV_PADDLE / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")

if "paddleocr" in MESIN:
    if not PY_PADDLE.exists():
        print("Membuat .venv-paddle ...")
        subprocess.run([sys.executable, "-m", "venv", str(VENV_PADDLE)], check=True)
    r = subprocess.run([str(PY_PADDLE), "-c", "import paddleocr"], capture_output=True)
    if r.returncode != 0:
        print("Memasang paddlepaddle + paddleocr (± 1 GB, beberapa menit)...")
        subprocess.run([str(PY_PADDLE), "-m", "pip", "install", "-q",
                        "paddlepaddle==3.3.1", "paddleocr==3.7.0"], check=True)
    r = subprocess.run([str(PY_PADDLE), "-c",
                        "import paddle, paddleocr; print(paddle.__version__, paddleocr.__version__)"],
                       capture_output=True, text=True)
    print("paddle / paddleocr:", r.stdout.strip() or r.stderr.strip()[-300:])

## 2. Audit resolusi citra

Resolusi menentukan kualitas OCR lebih dari mesin mana pun. Dokumen Tobacco3482 adalah
dokumen AS berukuran *letter* (lebar 8,5 inci), jadi perkiraan DPI = lebar piksel / 8,5.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src.utils import save_fig

manifest = pd.read_csv(MANIFEST, dtype={"id": str})   # id bisa diawali nol
ukuran = []
for p in manifest["path_citra"]:
    with Image.open(p) as im:        # hanya membaca header, cepat
        ukuran.append(im.size)
manifest["lebar"], manifest["tinggi"] = zip(*ukuran)
manifest["dpi_perkiraan"] = manifest["lebar"] / 8.5
display(manifest[["lebar", "tinggi", "dpi_perkiraan"]].describe().round(1))

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(manifest["dpi_perkiraan"], bins=40, color="#4C78A8")
ax.set_xlabel("perkiraan DPI (lebar / 8,5 inci)")
ax.set_ylabel("jumlah halaman")
ax.set_title("Resolusi citra Tobacco3482 (JPG Kaggle)")
save_fig(fig, "ocr_resolusi_citra")
plt.show()

## 3. Uji coba: kecepatan dan kewajaran keluaran

Setiap mesin dijalankan sebagai **subprocess** (`python -m src.ocr`), sehingga crash atau
bentrokan DLL di satu mesin tidak menjatuhkan kernel ini. Halaman uji coba tidak dibuang:
proses penuh nanti melewatinya.

In [ ]:
from collections import deque

def ada_cuda():
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                       capture_output=True, text=True)
    return r.stdout.strip() == "True"

GPU = ada_cuda()
print("EasyOCR memakai GPU:", GPU)

def jalankan(mesin, limit=None):
    py = str(PY_PADDLE) if mesin == "paddleocr" else sys.executable
    cmd = [py, "-m", "src.ocr", "--engine", mesin, "--manifest", str(MANIFEST),
           "--out", str(OUT)]
    if limit:
        cmd += ["--limit", str(limit)]
    if mesin == "easyocr" and GPU:
        cmd.append("--gpu")
    if mesin == "tesseract" and TESSERACT_CMD:
        cmd += ["--tesseract-cmd", TESSERACT_CMD]
    if mesin == "paddleocr":
        cmd += ["--paddle-models", PADDLE_MODEL, "--device", "cpu"]
    env = {**os.environ, "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True",
           "PYTHONIOENCODING": "utf-8"}
    ringkasan, ekor = None, deque(maxlen=40)
    with subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace") as proc:
        for line in proc.stdout:
            ekor.append(line)
            if line.startswith("RINGKASAN "):
                ringkasan = json.loads(line[len("RINGKASAN "):])
            elif line.startswith(f"[{mesin}]"):
                print(line, end="")
    if proc.returncode != 0 or ringkasan is None:
        print("".join(ekor))
        raise RuntimeError(f"{mesin} gagal (kode {proc.returncode}). Lihat keluaran di atas.")
    return ringkasan

def detik_per_halaman(mesin):
    # dari JSON, bukan dari ringkasan: run yang dilanjutkan hanya melihat sisanya
    d = [json.loads(p.read_text(encoding="utf-8"))["seconds"]
         for p in (OUT / mesin).glob("*.json") if not p.name.startswith("_")]
    return pd.Series(d).median() if d else float("nan")

uji = {}
for m in MESIN:
    print(f"\n=== {m}: uji coba {UJI_COBA} halaman")
    uji[m] = jalankan(m, UJI_COBA)

n = len(manifest)
est = pd.DataFrame({m: {"dtk/halaman (median)": round(detik_per_halaman(m), 2),
                        "perkiraan penuh (jam)": round(detik_per_halaman(m) * n / 3600, 2)}
                    for m in MESIN}).T
display(est)
print(f"Total perkiraan: {est['perkiraan penuh (jam)'].sum():.1f} jam (berurutan)")

In [ ]:
from src.ocr import load_texts

contoh = manifest["id"].iloc[0]
print(f"Dokumen {contoh} ({manifest['kelas'].iloc[0]}), 400 karakter pertama:\n")
print("--- QS-OCR (Tesseract 4 pada TIF) ---")
print(Path(manifest["path_teks"].iloc[0]).read_text(encoding="utf-8", errors="replace")[:400])
for m in MESIN:
    print(f"\n--- {m} ---")
    print(load_texts(OUT, m).get(str(contoh), "(belum ada)")[:400])

## 4. Proses penuh

Berjalan berurutan, satu mesin selesai baru mesin berikutnya. **Boleh diputus kapan saja**
(Kernel → Interrupt). Run All lagi melanjutkan dari halaman terakhir yang tersimpan.
Halaman yang gagal dicatat di `data/interim/ocr/<mesin>/_errors.txt` dan dicoba lagi pada
run berikutnya.

In [ ]:
if JALANKAN_PENUH:
    t0 = time.perf_counter()
    for m in MESIN:
        print(f"\n=== {m}: penuh")
        print(jalankan(m))
    print(f"\nselesai dalam {(time.perf_counter() - t0) / 3600:.2f} jam")
else:
    print("JALANKAN_PENUH = False — berhenti setelah uji coba.")

## 5. Proksi kualitas OCR

Tobacco3482 **tidak punya transkripsi acuan**, jadi *character error rate* tidak bisa
dihitung. Yang diukur di bawah menggambarkan keluaran, bukan menilai benar-salahnya:

- **kata/halaman** dan **halaman kosong**: mesin yang melewatkan blok teks terlihat di sini.
- **OOV**: bagian kata alfabetik (≥ 4 huruf) yang tidak ada di daftar kata bahasa Inggris
  `wordfreq`. Daftar itu longgar: salah eja yang kebetulan berupa kata web (mis. `tbe`)
  terhitung "dikenal", jadi angka ini **batas bawah** galat, bukan ukurannya.
- **Kesepakatan (Jaccard)** antar-mesin atas *himpunan* kata ≥ 3 huruf. Sengaja
  mengabaikan urutan, karena mesin berbasis kotak dan Tesseract berbeda urutan baca
  meski kata yang dibaca sama.

QS-OCR ikut sebagai baris acuan Fase 1.

In [ ]:
from functools import lru_cache
from itertools import combinations
from wordfreq import zipf_frequency
from src.ocr import jaccard, oov_share, words

@lru_cache(maxsize=None)
def dikenal(w):
    return zipf_frequency(w, "en") > 0

teks = {"qs-ocr": {str(i): Path(p).read_text(encoding="utf-8", errors="replace")
                   for i, p in zip(manifest["id"], manifest["path_teks"])}}
for m in MESIN:
    teks[m] = load_texts(OUT, m)

cakupan = {m: len(t) / len(manifest) for m, t in teks.items()}
print("cakupan dokumen:", {m: f"{c:.1%}" for m, c in cakupan.items()})
if any(c < 1 for c in cakupan.values()):
    print("PERINGATAN: ada mesin yang belum selesai; angka di bawah hanya untuk "
          "dokumen yang sudah ada. Jalankan bagian 4 sampai tuntas untuk angka final.")

baris = []
for i, kelas in zip(manifest["id"].astype(str), manifest["kelas"]):
    for m, t in teks.items():
        if i in t:
            baris.append({"id": i, "kelas": kelas, "mesin": m,
                          "n_kata": len(words(t[i])),
                          "oov": oov_share(t[i], dikenal)})
per_dok = pd.DataFrame(baris)

def ringkas(g):
    return pd.Series({
        "cakupan": cakupan[g.name],
        "kata/halaman (median)": g["n_kata"].median(),
        "halaman kosong (%)": 100 * (g["n_kata"] == 0).mean(),
        "OOV median": g["oov"].median(),
        "OOV rata-rata": g["oov"].mean(),
        "dtk/halaman": detik_per_halaman(g.name) if g.name != "qs-ocr" else float("nan"),
    })

ringkasan = per_dok.groupby("mesin").apply(ringkas).round(4)
display(ringkasan)

In [ ]:
nama = list(teks)
umum = set.intersection(*(set(t) for t in teks.values()))
kesepakatan = pd.DataFrame(1.0, index=nama, columns=nama)
for a, b in combinations(nama, 2):
    s = pd.Series([jaccard(teks[a][i], teks[b][i]) for i in umum]).median()
    kesepakatan.loc[a, b] = kesepakatan.loc[b, a] = round(s, 4)
print(f"median Jaccard himpunan kata, pada {len(umum)} dokumen yang dibaca semua mesin")
display(kesepakatan)

In [ ]:
oov_kelas = per_dok.pivot_table(index="kelas", columns="mesin", values="oov",
                                aggfunc="median").round(3)
print("OOV median per kelas (kelas dengan OOV tinggi adalah kandidat tempat mesin berbeda)")
display(oov_kelas)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
urut = nama
axes[0].boxplot([per_dok.loc[per_dok.mesin == m, "oov"].dropna() for m in urut],
                showfliers=False)
axes[0].set_ylabel("OOV per halaman")
axes[0].set_title("Bagian kata di luar daftar kata (batas bawah galat)")
axes[1].boxplot([per_dok.loc[per_dok.mesin == m, "n_kata"] for m in urut],
                showfliers=False)
axes[1].set_ylabel("kata per halaman")
axes[1].set_title("Jumlah kata yang terbaca")
for ax in axes:
    ax.set_xticks(range(1, len(urut) + 1), urut)   # bukan labels=: berganti nama di mpl 3.9
    ax.grid(alpha=0.3)
save_fig(fig, "ocr_proksi_kualitas")
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(kesepakatan.values, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(nama)), nama, rotation=30)
ax.set_yticks(range(len(nama)), nama)
for (r, c), v in pd.DataFrame(kesepakatan.values).stack().items():
    ax.text(c, r, f"{v:.2f}", ha="center", va="center",
            color="white" if v < 0.6 else "black")
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Kesepakatan antar-mesin (median Jaccard)")
save_fig(fig, "ocr_kesepakatan_antar_mesin")
plt.show()

## Simpan dan salin ke catatan

Tidak ada vonis otomatis di sini — pelajaran dari notebook 07–09 adalah bahwa ambang
biner yang dipasang sebelum melihat data cenderung menyesatkan. Angka di bawah dibawa ke
`vault/30-eksperimen/13-ocr-multi-mesin.md`, dan penafsirannya ditulis di sana.

In [ ]:
rep = ROOT / "reports"
per_dok.to_csv(rep / "ocr_proksi_per_dokumen.csv", index=False)
ringkasan.to_csv(rep / "ocr_ringkasan.csv")
kesepakatan.to_csv(rep / "ocr_kesepakatan.csv")
oov_kelas.to_csv(rep / "ocr_oov_per_kelas.csv")
meta = {m: json.loads((OUT / m / "_meta.json").read_text()) for m in MESIN
        if (OUT / m / "_meta.json").exists()}
(rep / "ocr_meta.json").write_text(json.dumps(meta, indent=1))
print("disimpan: reports/ocr_proksi_per_dokumen.csv, ocr_ringkasan.csv, "
      "ocr_kesepakatan.csv, ocr_oov_per_kelas.csv, ocr_meta.json\n")

print("--- untuk disalin ke catatan ---")
print(ringkasan.to_string())
print()
print(kesepakatan.to_string())
print()
print(json.dumps(meta, indent=1))